In [ ]:
# BERNOULLI DISTRIBUTION
import numpy as np

# WE'RE GOING TO ASK PYTHON TO RANDOMLY GENERATE OUTCOMES.
# BUT COMPUTERS DON'T PRODUCE TRULY RANDOM NUMBERS IN THE ORDINARY SENSE. THEY USE A PROCESS CALLED PSEUDO-RANDOM GENERATION.
# THE SEED ALLOWS US TO MAKE THE RANDOM SEQUENCE REPRODUCIBLE. SO, EVRYTIME WE RUN THIS CODE, WE WILL GET THE SAME RANDOM NUMBERS. THIS IS USEFUL FOR DEBUGGING AND TESTING.
np.random.seed(42)

p = 0.3

# BUT A BERNOULLI EXPERIMENT IS ACTUALLY JUST A BINOMIAL EXPERIMENT WITH ONE TRIAL. WHICH MEANS WE CAN USE THE BINOMIAL FUNCTION TO SIMULATE A BERNOULLI EXPERIMENT. THE BINOMIAL FUNCTION TAKES THREE ARGUMENTS: THE NUMBER OF TRIALS, THE PROBABILITY OF SUCCESS, AND THE NUMBER OF EXPERIMENTS TO RUN. IN THIS CASE, WE'RE GOING TO RUN 1000 EXPERIMENTS.
customers = np.random.binomial(1, p, 1000)

print(customers[:20])  # FIRST 20 OUTCOMES
print("Average:", customers.mean())  # IT ISN'T EXACTLY 0.3. BUT IT'S CLOSE.
# THIS IS THE INTUITION BEHIND THE LAW OF LARGE NUMBERS. AS WE PERFORM MORE AND MORE INDEPENDENT TRIALS, THE OBSERVED AVERAGE TENDS TO APPROACH THE EXPECTED VALUE/PROBABILITY.

[0 1 1 0 0 0 0 1 0 1 0 1 1 0 0 0 0 0 0 0]
Average: 0.288


In [ ]:
# BINOMIAL DISTRIBUTION
p = 0.3
n = 100

purchases = np.random.binomial(n, p, 1000)

print(purchases[:20])
print("Average purchases:", purchases.mean())

[26 30 35 33 34 32 32 35 27 30 26 41 37 22 32 37 26 31 36 22]
Average purchases: 30.127


In [ ]:
# POISSON DISTRIBUTION
import numpy as np

np.random.seed(42)

lambda_ = 5  # AVERAGE = 5 ERRORS PER HOUR.

# SIMULATE 1,000 ONE-HOUR PERIODS WHERE THE AVERAGE NUMBER OF ERRORS IS 5.
errors = np.random.poisson(lambda_, 1000)

print(errors[:20])
print("Average errors:", errors.mean())

[5 4 4 5 5 3 5 4 6 7 2 5 5 6 4 6 6 1 7 2]
Average errors: 4.985


In [2]:
# NORMAL DISTRIBUTION
import numpy as np

np.random.seed(42)

mean = 170
std = 10

# SIMULATE 1000 PEOPLE WHOSE HEIGHTS APPROXIMATELY FOLLOW A NORMAL DISTRIBUTION WITH:
# MEAN = 170, STANDARD DEVIATION = 10
heights = np.random.normal(mean, std, 1000)

print(heights[:10])

print("Average:", heights.mean())
print("Standard Deviation:", heights.std())

[174.96714153 168.61735699 176.47688538 185.23029856 167.65846625
 167.65863043 185.79212816 177.67434729 165.30525614 175.42560044]
Average: 170.19332055822326
Standard Deviation: 9.787262077473542


In [3]:
pip install xlrd

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ==============================================================
# REVERSE ENGINEERING BASED DATA RECONSTRUCTION
# ==============================================================


import pandas as pd
import numpy as np

df = pd.read_excel("Kaptai_Solar_PV_Operational_Dataset_2020.xls")


df["Date"] = pd.to_datetime(df["Date"])


# ==============================================================
# 1. WS2 EAST REVERSE ENGINEERING
# ==============================================================


east_ratio = (
    df.loc[df["WS2_East"].notna(), "WS2_East"]
    / df.loc[df["WS2_East"].notna(), "WS1_East"]
)


east_factor = east_ratio.median()


missing_east = df["WS2_East"].isna()


df.loc[missing_east, "WS2_East"] = df.loc[missing_east, "WS1_East"] * east_factor


# ==============================================================
# 2. WS2 WEST REVERSE ENGINEERING
# ==============================================================


west_ratio = (
    df.loc[df["WS2_West"].notna(), "WS2_West"]
    / df.loc[df["WS2_West"].notna(), "WS1_West"]
)


west_factor = west_ratio.median()


missing_west = df["WS2_West"].isna()


df.loc[missing_west, "WS2_West"] = df.loc[missing_west, "WS1_West"] * west_factor


# ==============================================================
# 3. PR REVERSE ENGINEERING
# ==============================================================


df["PR"] = df["Output_Energy"] / df["Input_Energy"]


# ==============================================================
# 4. VALIDATION
# ==============================================================


print("Missing values after reconstruction:")


print(df.isnull().sum())


assert df.isnull().sum().sum() == 0


# Save

df.to_excel("Kaptai_Solar_PV_Operational_Dataset_Cleaned.xlsx", index=False)

# ==============================================================
# FINAL SPHI PIPELINE
#
# PART 1/3
#
# Data preparation and Autoencoder configuration
# ==============================================================


import os
import random
import warnings


import numpy as np
import pandas as pd


import matplotlib.pyplot as plt


import tensorflow as tf


from sklearn.preprocessing import MinMaxScaler


from tensorflow.keras.models import Model


from tensorflow.keras.layers import Input, Dense, Dropout


from tensorflow.keras.optimizers import Adam


from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings("ignore")


# ==============================================================
# Reproducibility
# ==============================================================


SEED = 42


np.random.seed(SEED)

random.seed(SEED)

tf.random.set_seed(SEED)


# ==============================================================
# Directories
# ==============================================================


RESULT_DIR = "SPHI_Final_Result"

FIG_DIR = os.path.join(RESULT_DIR, "Figures")


TABLE_DIR = os.path.join(RESULT_DIR, "Tables")


os.makedirs(FIG_DIR, exist_ok=True)

os.makedirs(TABLE_DIR, exist_ok=True)


# ==============================================================
# Load reconstructed dataset
# ==============================================================


df = pd.read_excel("Kaptai_Solar_PV_Operational_Dataset_Cleaned.xlsx")


df["Date"] = pd.to_datetime(df["Date"])


df = df.sort_values("Date").reset_index(drop=True)


print(df.shape)

print(df.head())


# ==============================================================
# Data quality verification
# ==============================================================


quality = pd.DataFrame(
    {
        "Parameter": [
            "Samples",
            "Variables",
            "Missing values",
            "Duplicate rows",
            "Start date",
            "End date",
        ],
        "Value": [
            len(df),
            len(df.columns),
            df.isnull().sum().sum(),
            df.duplicated().sum(),
            df.Date.min(),
            df.Date.max(),
        ],
    }
)


quality.to_excel(os.path.join(TABLE_DIR, "Table_I_Data_Quality.xlsx"), index=False)


assert df.isnull().sum().sum() == 0


# ==============================================================
# Final model features
# ==============================================================


model_features = ["WS1_East", "WS2_East", "WS1_West", "WS2_West", "Radiation_kWh"]


X = df[model_features].copy()


# ==============================================================
# Feature statistics
# ==============================================================


X.describe().T.to_excel(os.path.join(TABLE_DIR, "Table_II_Feature_Statistics.xlsx"))


# ==============================================================
# IMPORTANT:
#
# Use all operational days
#
# No PR/output filtering
#
# ==============================================================


X_normal = X.copy()


print("Training samples:", len(X_normal))


# ==============================================================
# Chronological split
# ==============================================================


split = int(len(X_normal) * 0.8)


X_train = X_normal.iloc[:split]


X_validation = X_normal.iloc[split:]


pd.DataFrame(
    {
        "Dataset": ["Training", "Validation"],
        "Samples": [len(X_train), len(X_validation)],
    }
).to_excel(os.path.join(TABLE_DIR, "Table_III_Data_Split.xlsx"), index=False)


# ==============================================================
# MinMax Scaling
# ==============================================================


scaler = MinMaxScaler()


X_train_scaled = scaler.fit_transform(X_train)


X_validation_scaled = scaler.transform(X_validation)


X_all_scaled = scaler.transform(X)


assert np.isnan(X_train_scaled).sum() == 0

assert np.isnan(X_validation_scaled).sum() == 0

assert np.isnan(X_all_scaled).sum() == 0


# ==============================================================
# Autoencoder
#
# 5-16-4-16-5
# ==============================================================


inputs = Input(shape=(len(model_features),))


x = Dense(16, activation="relu")(inputs)


x = Dropout(0.1)(x)


latent = Dense(4, activation="relu", name="latent")(x)


x = Dense(16, activation="relu")(latent)


outputs = Dense(len(model_features), activation="linear")(x)


autoencoder = Model(inputs, outputs)


autoencoder.compile(optimizer=Adam(learning_rate=0.0005), loss="mse")


autoencoder.summary()


# ==============================================================
# Model table
# ==============================================================


pd.DataFrame(
    {
        "Parameter": [
            "Features",
            "Architecture",
            "Latent dimension",
            "Scaler",
            "Optimizer",
            "Loss",
        ],
        "Value": [str(model_features), "5-16-4-16-5", 4, "MinMaxScaler", "Adam", "MSE"],
    }
).to_excel(os.path.join(TABLE_DIR, "Table_IV_Model_Configuration.xlsx"), index=False)


print("PART 1 COMPLETED")

# ==============================================================
# FINAL SPHI PIPELINE
#
# PART 2/3
#
# Autoencoder Training + Reconstruction Analysis
# ==============================================================


# ==============================================================
# 15. TRAINING CALLBACKS
# ==============================================================


early_stop = EarlyStopping(monitor="val_loss", patience=80, restore_best_weights=True)


reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=20, min_lr=1e-7)


# ==============================================================
# 16. AUTOENCODER TRAINING
# ==============================================================


history = autoencoder.fit(
    X_train_scaled,
    X_train_scaled,
    validation_data=(X_validation_scaled, X_validation_scaled),
    epochs=500,
    batch_size=16,
    callbacks=[early_stop, reduce_lr],
    verbose=1,
)


# ==============================================================
# 17. TRAINING PERFORMANCE TABLE
# ==============================================================


training_table = pd.DataFrame(
    {
        "Parameter": [
            "Completed Epochs",
            "Minimum Training Loss",
            "Minimum Validation Loss",
        ],
        "Value": [
            len(history.history["loss"]),
            min(history.history["loss"]),
            min(history.history["val_loss"]),
        ],
    }
)


training_table.to_excel(
    os.path.join(TABLE_DIR, "Table_V_Training_Performance.xlsx"), index=False
)


print(training_table)


# ==============================================================
# 18. TRAINING CURVE
# ==============================================================


plt.figure(figsize=(8, 5), dpi=300)


plt.plot(history.history["loss"], linewidth=2, label="Training Loss")


plt.plot(history.history["val_loss"], linewidth=2, label="Validation Loss")


plt.xlabel("Epoch")


plt.ylabel("MSE Loss")


plt.title("Autoencoder Training Performance")


plt.grid(alpha=0.3)


plt.legend()


plt.tight_layout()


plt.savefig(
    os.path.join(FIG_DIR, "Figure_Training_Curve.png"), dpi=300, bbox_inches="tight"
)


plt.close()


# ==============================================================
# 19. SAVE MODEL
# ==============================================================


autoencoder.save(os.path.join(RESULT_DIR, "SPHI_Autoencoder_Model.keras"))


# ==============================================================
# 20. VALIDATION RECONSTRUCTION
# ==============================================================


X_validation_prediction = autoencoder.predict(X_validation_scaled, verbose=0)


# ==============================================================
# 21. RECONSTRUCTION PERFORMANCE
# ==============================================================


from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

overall_mse = mean_squared_error(X_validation_scaled, X_validation_prediction)


overall_rmse = np.sqrt(overall_mse)


overall_mae = mean_absolute_error(X_validation_scaled, X_validation_prediction)


# ==============================================================
# 22. FEATURE-WISE METRICS
# ==============================================================


feature_results = []


for i, feature in enumerate(model_features):

    mse = mean_squared_error(X_validation_scaled[:, i], X_validation_prediction[:, i])

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(X_validation_scaled[:, i], X_validation_prediction[:, i])

    r2 = r2_score(X_validation_scaled[:, i], X_validation_prediction[:, i])

    feature_results.append(
        {"Feature": feature, "MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}
    )


feature_table = pd.DataFrame(feature_results)


mean_r2 = feature_table["R2"].mean()


performance_table = pd.DataFrame(
    {
        "Metric": ["MSE", "RMSE", "MAE", "Mean Feature-wise R2"],
        "Value": [overall_mse, overall_rmse, overall_mae, mean_r2],
    }
)


performance_table.to_excel(
    os.path.join(TABLE_DIR, "Table_VI_Reconstruction_Performance.xlsx"), index=False
)


feature_table.to_excel(
    os.path.join(TABLE_DIR, "Table_VII_Featurewise_Performance.xlsx"), index=False
)


print("\nOverall Performance")

print(performance_table)


print("\nFeature Performance")

print(feature_table)


# ==============================================================
# 23. VALIDATION RECONSTRUCTION ERROR
# ==============================================================


validation_error = np.mean(
    np.square(X_validation_scaled - X_validation_prediction), axis=1
)


# ==============================================================
# 24. ROBUST THRESHOLD USING MAD
# ==============================================================


median_error = np.median(validation_error)


mad_error = np.median(np.abs(validation_error - median_error))


threshold = median_error + 3 * mad_error


threshold_table = pd.DataFrame(
    {
        "Parameter": ["Median Error", "MAD", "Threshold (Median+3MAD)"],
        "Value": [median_error, mad_error, threshold],
    }
)


threshold_table.to_excel(
    os.path.join(TABLE_DIR, "Table_VIII_Robust_Threshold.xlsx"), index=False
)


print("Final Threshold:", threshold)


# ==============================================================
# 25. ERROR DISTRIBUTION
# ==============================================================


plt.figure(figsize=(7, 5), dpi=300)


plt.hist(validation_error, bins=30)


plt.axvline(threshold, linestyle="--", linewidth=2, label="Median + 3MAD")


plt.xlabel("Reconstruction Error")


plt.ylabel("Frequency")


plt.title("Normal Operation Error Distribution")


plt.grid(alpha=0.3)


plt.legend()


plt.tight_layout()


plt.savefig(
    os.path.join(FIG_DIR, "Figure_Error_Distribution.png"), dpi=300, bbox_inches="tight"
)


plt.close()


# ==============================================================
# 26. FULL DATASET RECONSTRUCTION
# ==============================================================


X_all_prediction = autoencoder.predict(X_all_scaled, verbose=0)


full_error = np.mean(np.square(X_all_scaled - X_all_prediction), axis=1)


results = df.copy()


results["Reconstruction_Error"] = full_error


results["Threshold"] = threshold


# ==============================================================
# 27. DEVIATION IDENTIFICATION
# ==============================================================


results["Deviation_Status"] = np.where(
    results["Reconstruction_Error"] > threshold, "Deviation", "Normal"
)


# ==============================================================
# 28. DEVIATION SUMMARY
# ==============================================================


deviation_count = (results["Deviation_Status"] == "Deviation").sum()


summary = pd.DataFrame(
    {
        "Parameter": ["Total Samples", "Detected Deviations", "Deviation Percentage"],
        "Value": [len(results), deviation_count, deviation_count / len(results) * 100],
    }
)


summary.to_excel(
    os.path.join(TABLE_DIR, "Table_IX_Deviation_Summary.xlsx"), index=False
)


print(summary)


# ==============================================================
# 29. RECONSTRUCTION ERROR TIME SERIES
# ==============================================================


plt.figure(figsize=(12, 5), dpi=300)


plt.plot(
    results["Date"],
    results["Reconstruction_Error"],
    linewidth=1.8,
    label="Reconstruction Error",
)


plt.axhline(threshold, linestyle="--", linewidth=2, label="Median + 3MAD")


deviation_points = results[results["Deviation_Status"] == "Deviation"]


plt.scatter(
    deviation_points["Date"],
    deviation_points["Reconstruction_Error"],
    s=45,
    label="Deviation",
)


plt.xlabel("Date")


plt.ylabel("Reconstruction Error")


plt.title("PV Operational Deviation Detection")


plt.xticks(rotation=45)


plt.grid(alpha=0.3)


plt.legend()


plt.tight_layout()


plt.savefig(
    os.path.join(FIG_DIR, "Figure_Reconstruction_Error.png"),
    dpi=300,
    bbox_inches="tight",
)


plt.close()


# ==============================================================
# 30. TOP DEVIATION EVENTS
# ==============================================================


top10 = results.sort_values("Reconstruction_Error", ascending=False).head(10)


top10[
    ["Date", "Reconstruction_Error", "Output_Energy", "PR", "Deviation_Status"]
].to_excel(os.path.join(TABLE_DIR, "Table_X_Top_Deviation_Events.xlsx"), index=False)


# ==============================================================
# 31. SAVE RESULTS
# ==============================================================


results.to_excel(
    os.path.join(RESULT_DIR, "SPHI_Reconstruction_Results.xlsx"), index=False
)


print("\nPART 2 COMPLETED SUCCESSFULLY")

# ==============================================================
# FINAL SPHI PIPELINE
#
# PART 3/3
#
# Final SPHI Calculation + Validation + Sensitivity Analysis
# ==============================================================


# ==============================================================
# 32. FINAL CALIBRATED SPHI CALCULATION
# ==============================================================


# SPHI formulation:
#
# SPHI = 100 * exp(-Reconstruction_Error/(2*Threshold))
#
# Threshold = Median Error + 3MAD
#
# ==============================================================


SPHI_FACTOR = 2


sphi_scale = SPHI_FACTOR * threshold


results["SPHI"] = 100 * np.exp(-results["Reconstruction_Error"] / sphi_scale)


results["SPHI"] = np.clip(results["SPHI"], 0, 100)


# ==============================================================
# 33. HEALTH CLASSIFICATION
# ==============================================================


def health_class(value):

    if value >= 80:

        return "Healthy"

    elif value >= 60:

        return "Warning"

    else:

        return "Degraded"


results["Health_Status"] = results["SPHI"].apply(health_class)


# ==============================================================
# 34. SPHI METHODOLOGY TABLE
# ==============================================================


sphi_method = pd.DataFrame(
    {
        "Parameter": ["SPHI Equation", "Scaling Factor", "Threshold Method"],
        "Value": [
            "100*exp(-Reconstruction_Error/(Factor*Threshold))",
            SPHI_FACTOR,
            "Median + 3MAD",
        ],
    }
)


sphi_method.to_excel(
    os.path.join(TABLE_DIR, "Table_XI_SPHi_Methodology.xlsx"), index=False
)


# ==============================================================
# 35. SPHI SUMMARY TABLE
# ==============================================================


sphi_summary = pd.DataFrame(
    {
        "Parameter": [
            "Average SPHI",
            "Maximum SPHI",
            "Minimum SPHI",
            "Healthy Days",
            "Warning Days",
            "Degraded Days",
        ],
        "Value": [
            results["SPHI"].mean(),
            results["SPHI"].max(),
            results["SPHI"].min(),
            (results["Health_Status"] == "Healthy").sum(),
            (results["Health_Status"] == "Warning").sum(),
            (results["Health_Status"] == "Degraded").sum(),
        ],
    }
)


sphi_summary.to_excel(
    os.path.join(TABLE_DIR, "Table_XII_SPHi_Summary.xlsx"), index=False
)


print("\nSPHI Summary")

print(sphi_summary)


# ==============================================================
# 36. SPHI TIME SERIES
# ==============================================================


plt.figure(figsize=(12, 5), dpi=300)


plt.plot(results["Date"], results["SPHI"], linewidth=2, label="SPHI")


plt.axhline(80, linestyle="--", linewidth=1.5, label="Healthy Limit")


plt.axhline(60, linestyle="--", linewidth=1.5, label="Warning Limit")


plt.xlabel("Date")


plt.ylabel("SPHI")


plt.title("Solar Plant Health Index Variation")


plt.xticks(rotation=45)


plt.grid(alpha=0.3)


plt.legend()


plt.tight_layout()


plt.savefig(
    os.path.join(FIG_DIR, "Figure_SPHi_TimeSeries.png"), dpi=300, bbox_inches="tight"
)


plt.close()


# ==============================================================
# 37. MONTHLY SPHI AND PERFORMANCE ANALYSIS
# ==============================================================


monthly_analysis = (
    results.groupby("Month")
    .agg(
        Mean_SPHI=("SPHI", "mean"),
        Mean_PR=("PR", "mean"),
        Mean_Error=("Reconstruction_Error", "mean"),
        Healthy_Days=("Health_Status", lambda x: (x == "Healthy").sum()),
        Warning_Days=("Health_Status", lambda x: (x == "Warning").sum()),
        Degraded_Days=("Health_Status", lambda x: (x == "Degraded").sum()),
    )
    .reset_index()
)


monthly_analysis.to_excel(
    os.path.join(TABLE_DIR, "Table_XIII_Monthly_SPHi_PR_Analysis.xlsx"), index=False
)


print(monthly_analysis)


# ==============================================================
# 38. MONTHLY SPHI FIGURE
# ==============================================================


plt.figure(figsize=(10, 5), dpi=300)


plt.bar(monthly_analysis["Month"], monthly_analysis["Mean_SPHI"])


plt.xlabel("Month")


plt.ylabel("Average SPHI")


plt.title("Monthly Solar Plant Health Index")


plt.xticks(rotation=45)


plt.grid(alpha=0.3)


plt.tight_layout()


plt.savefig(
    os.path.join(FIG_DIR, "Figure_Monthly_SPHi.png"), dpi=300, bbox_inches="tight"
)


plt.close()


# ==============================================================
# 39. ENERGY YIELD CALCULATION
# ==============================================================


results["Energy_Yield"] = results["Output_Energy"] / results["Radiation_kWh"]


results.replace([np.inf, -np.inf], np.nan, inplace=True)


results.dropna(subset=["Energy_Yield"], inplace=True)


# ==============================================================
# 40. SPHI VALIDATION
# ==============================================================


from scipy.stats import pearsonr

corr_pr, p_pr = pearsonr(results["SPHI"], results["PR"])


corr_energy, p_energy = pearsonr(results["SPHI"], results["Energy_Yield"])


validation_table = pd.DataFrame(
    {
        "Relationship": ["SPHI vs PR", "SPHI vs Energy Yield"],
        "Pearson_R": [corr_pr, corr_energy],
        "p_value": [p_pr, p_energy],
    }
)


validation_table.to_excel(
    os.path.join(TABLE_DIR, "Table_XIV_SPHi_Validation.xlsx"), index=False
)


print("\nValidation Results")

print(validation_table)


# ==============================================================
# 41. THRESHOLD SENSITIVITY ANALYSIS
# ==============================================================


median_error = np.median(results["Reconstruction_Error"])


mad_error = np.median(np.abs(results["Reconstruction_Error"] - median_error))


threshold_results = []


for multiplier in [2, 3, 4]:

    test_threshold = median_error + multiplier * mad_error

    deviation_count = (results["Reconstruction_Error"] > test_threshold).sum()

    threshold_results.append(
        {
            "MAD_Multiplier": multiplier,
            "Threshold": test_threshold,
            "Deviation_Days": deviation_count,
            "Deviation_Percentage": deviation_count / len(results) * 100,
        }
    )


threshold_sensitivity = pd.DataFrame(threshold_results)


threshold_sensitivity.to_excel(
    os.path.join(TABLE_DIR, "Table_XV_Threshold_Sensitivity.xlsx"), index=False
)


print(threshold_sensitivity)


# ==============================================================
# 42. DEGRADATION EVENT IDENTIFICATION
# ==============================================================


degradation_events = results.sort_values("Reconstruction_Error", ascending=False).head(
    20
)


degradation_events[
    [
        "Date",
        "SPHI",
        "Reconstruction_Error",
        "PR",
        "Radiation_kWh",
        "Output_Energy",
        "Health_Status",
    ]
].to_excel(os.path.join(TABLE_DIR, "Table_XVI_Degradation_Events.xlsx"), index=False)


# ==============================================================
# 43. FINAL EXPORT
# ==============================================================


results.to_excel(os.path.join(RESULT_DIR, "FINAL_SPHI_RESULTS.xlsx"), index=False)


print("\nFINAL SPHI ANALYSIS COMPLETED SUCCESSFULLY")

# ==============================================================
# IEEE CONFERENCE PAPER
# RESULTS AND DISCUSSION GENERATION
#
# Final Manuscript Version
#
# ==============================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from IPython.display import display


from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


from scipy.stats import pearsonr

# ==============================================================
# PREPARE FINAL DATA
# ==============================================================


paper_results = results.copy()


# ==============================================================
# TABLE I
# AUTOENCODER CONFIGURATION AND PERFORMANCE
# ==============================================================


X_prediction = autoencoder.predict(X_validation_scaled, verbose=0)


mse = mean_squared_error(X_validation_scaled, X_prediction)


rmse = np.sqrt(mse)


mae = mean_absolute_error(X_validation_scaled, X_prediction)


r2 = r2_score(X_validation_scaled, X_prediction)


table1 = pd.DataFrame(
    {
        "Parameter": [
            "Architecture",
            "Input features",
            "Latent dimension",
            "Optimizer",
            "Loss function",
            "RMSE",
            "MAE",
            "R²",
        ],
        "Value": [
            "5-16-4-16-5",
            len(model_features),
            4,
            "Adam",
            "MSE",
            round(rmse, 5),
            round(mae, 5),
            round(r2, 5),
        ],
    }
)


print("\nTABLE I")
print("Autoencoder Configuration and Reconstruction Performance")

display(table1)


# ==============================================================
# FIGURE 1
# AUTOENCODER TRAINING CURVE
# ==============================================================


plt.figure(figsize=(7, 4), dpi=300)


plt.plot(history.history["loss"], linewidth=2, label="Training Loss")


plt.plot(history.history["val_loss"], linewidth=2, label="Validation Loss")


plt.xlabel("Epoch")


plt.ylabel("MSE Loss")


plt.title("Autoencoder Training and Validation Performance")


plt.grid(alpha=0.3)

plt.legend()

plt.tight_layout()

plt.show()


# ==============================================================
# TABLE II
# THRESHOLD SENSITIVITY
# ==============================================================


table2 = threshold_sensitivity.copy()


table2.columns = [
    "MAD Multiplier",
    "Threshold",
    "Deviation Days",
    "Deviation Percentage",
]


print("\nTABLE II")

print("Threshold Sensitivity Analysis")

display(table2)


# ==============================================================
# FIGURE 2
# RECONSTRUCTION ERROR WITH THRESHOLD
# ==============================================================


plt.figure(figsize=(10, 4), dpi=300)


plt.plot(
    paper_results["Date"],
    paper_results["Reconstruction_Error"],
    linewidth=1.5,
    label="Reconstruction Error",
)


plt.axhline(threshold, linestyle="--", linewidth=2, label="3MAD Threshold")


deviation_points = paper_results[paper_results["Deviation_Status"] == "Deviation"]


plt.scatter(
    deviation_points["Date"],
    deviation_points["Reconstruction_Error"],
    s=35,
    label="Detected Deviation",
)


plt.xlabel("Date")


plt.ylabel("Reconstruction Error")


plt.title("Deviation Detection Using Reconstruction Error")


plt.xticks(rotation=45)

plt.grid(alpha=0.3)

plt.legend()

plt.tight_layout()

plt.show()


# ==============================================================
# TABLE III
# SPHI HEALTH ASSESSMENT
# ==============================================================


table3 = pd.DataFrame(
    {
        "Parameter": [
            "Average SPHI",
            "Maximum SPHI",
            "Minimum SPHI",
            "Healthy Days",
            "Warning Days",
            "Degraded Days",
        ],
        "Value": [
            round(paper_results["SPHI"].mean(), 3),
            round(paper_results["SPHI"].max(), 3),
            round(paper_results["SPHI"].min(), 3),
            (paper_results["Health_Status"] == "Healthy").sum(),
            (paper_results["Health_Status"] == "Warning").sum(),
            (paper_results["Health_Status"] == "Degraded").sum(),
        ],
    }
)


print("\nTABLE III")

print("SPHI Health Assessment")

display(table3)


# ==============================================================
# FIGURE 3
# ANNUAL SPHI TREND
# ==============================================================


plt.figure(figsize=(10, 4), dpi=300)


plt.plot(paper_results["Date"], paper_results["SPHI"], linewidth=2, label="SPHI")


plt.axhline(80, linestyle="--", label="Healthy limit")


plt.axhline(60, linestyle="--", label="Warning limit")


plt.xlabel("Date")


plt.ylabel("SPHI")


plt.title("Solar Plant Health Index Variation")


plt.xticks(rotation=45)

plt.grid(alpha=0.3)

plt.legend()

plt.tight_layout()

plt.show()


# ==============================================================
# TABLE IV
# MONTHLY SPHI PERFORMANCE
# ==============================================================


table4 = (
    paper_results.groupby("Month")
    .agg(
        Mean_SPHI=("SPHI", "mean"),
        Mean_PR=("PR", "mean"),
        Degraded_Days=("Health_Status", lambda x: (x == "Degraded").sum()),
    )
    .reset_index()
)


print("\nTABLE IV")

print("Monthly SPHI Performance")

display(table4)


# ==============================================================
# FIGURE 4
# MONTHLY SPHI
# ==============================================================


plt.figure(figsize=(9, 4), dpi=300)


plt.bar(table4["Month"], table4["Mean_SPHI"])


plt.xlabel("Month")


plt.ylabel("Average SPHI")


plt.title("Monthly Solar Plant Health Index")


plt.xticks(rotation=45)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()


# ==============================================================
# TABLE V
# REPRESENTATIVE DEGRADATION EVENTS
# ==============================================================


table5 = paper_results.sort_values("Reconstruction_Error", ascending=False).head(10)[
    ["Date", "SPHI", "Reconstruction_Error", "PR", "Output_Energy", "Health_Status"]
]


print("\nTABLE V")

print("Representative Degradation Events")

display(table5)


# ==============================================================
# FIGURE 5
# SPHI VALIDATION
# ==============================================================


fig, ax = plt.subplots(1, 2, figsize=(11, 4), dpi=300)


# SPHI vs PR


ax[0].scatter(paper_results["SPHI"], paper_results["PR"], s=35)


ax[0].set_xlabel("SPHI")


ax[0].set_ylabel("Performance Ratio")


ax[0].set_title("SPHI vs PR")


ax[0].grid(alpha=0.3)


# SPHI vs Energy Yield


ax[1].scatter(paper_results["SPHI"], paper_results["Energy_Yield"], s=35)


ax[1].set_xlabel("SPHI")


ax[1].set_ylabel("Energy Yield")


ax[1].set_title("SPHI vs Energy Yield")


ax[1].grid(alpha=0.3)


plt.tight_layout()

plt.show()


# ==============================================================
# VALIDATION STATISTICS
# ==============================================================


corr_pr, p_pr = pearsonr(paper_results["SPHI"], paper_results["PR"])


corr_energy, p_energy = pearsonr(paper_results["SPHI"], paper_results["Energy_Yield"])


validation = pd.DataFrame(
    {
        "Relationship": ["SPHI vs PR", "SPHI vs Energy Yield"],
        "Pearson_R": [corr_pr, corr_energy],
        "p_value": [p_pr, p_energy],
    }
)


print("\nVALIDATION STATISTICS")

display(validation)


print("\nIEEE MANUSCRIPT RESULTS GENERATION COMPLETED")

In [ ]:
# PROVIDED BY CODEX

# ==============================================================
# CORRECTED SPHI PIPELINE
#
# Fixes:
# 1. SPHI equation matches the research document.
# 2. Training uses an unsupervised normal-reference subset.
# 3. Independent training, calibration, and test datasets.
# 4. Multivariable WS2 reconstruction with validation.
# 5. Reconstructed WS2 observations are excluded from assessment.
# 6. Threshold sensitivity uses calibration errors only.
# 7. Results are called candidate operational deviations.
# ==============================================================


# ==============================================================
# 1. IMPORTS
# ==============================================================

import os
import random
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from IPython.display import display

from scipy.stats import pearsonr

from sklearn.ensemble import (
    IsolationForest,
    RandomForestRegressor,
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from sklearn.model_selection import (
    KFold,
    cross_val_predict,
)

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
)

from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Input,
)

from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")


# ==============================================================
# 2. REPRODUCIBILITY
# ==============================================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)


# ==============================================================
# 3. CONFIGURATION
# ==============================================================

INPUT_FILE = "Kaptai_Solar_PV_Operational_Dataset_2020.xls"

RESULT_DIR = "SPHI_Final_Result"
FIG_DIR = os.path.join(RESULT_DIR, "Figures")
TABLE_DIR = os.path.join(RESULT_DIR, "Tables")

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

TRAIN_FRACTION = 0.60
CALIBRATION_FRACTION = 0.20
TEST_FRACTION = 0.20

NORMAL_CONTAMINATION = 0.10
MAD_MULTIPLIER = 3.0


# ==============================================================
# 4. LOAD AND VALIDATE DATA
# ==============================================================

df = pd.read_excel(
    INPUT_FILE,
    engine="xlrd",
)

required_columns = [
    "Date",
    "WS1_East",
    "WS2_East",
    "WS1_West",
    "WS2_West",
    "Radiation_kWh",
    "Input_Energy",
    "Output_Energy",
    "PR",
]

missing_columns = sorted(set(required_columns) - set(df.columns))

if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce",
)

if df["Date"].isna().any():
    raise ValueError("Invalid Date values were detected.")

df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True,
)

df = (
    df.sort_values("Date")
    .drop_duplicates(subset=["Date"], keep="first")
    .reset_index(drop=True)
)


# ==============================================================
# 5. CHRONOLOGICAL PARTITION DEFINITION
#
# The partitions are defined before learning reconstruction
# models, scalers, or the autoencoder.
#
# 60% = Training
# 20% = Threshold calibration
# 20% = Independent test
# ==============================================================

number_of_rows = len(df)

train_end = int(number_of_rows * TRAIN_FRACTION)

calibration_end = int(number_of_rows * (TRAIN_FRACTION + CALIBRATION_FRACTION))

df["Evaluation_Set"] = "Unseen Test"

df.loc[
    df.index < train_end,
    "Evaluation_Set",
] = "Training"

df.loc[
    (df.index >= train_end) & (df.index < calibration_end),
    "Evaluation_Set",
] = "Threshold Calibration"


# ==============================================================
# 6. PERFORMANCE RATIO RECONSTRUCTION
#
# PR is reconstructed only where it is missing.
# Existing valid PR measurements are preserved.
# ==============================================================

df["PR_WasMissing"] = df["PR"].isna()

valid_pr_reconstruction = (
    df["PR"].isna()
    & df["Output_Energy"].notna()
    & df["Input_Energy"].notna()
    & (df["Input_Energy"] > 0)
)

df.loc[
    valid_pr_reconstruction,
    "PR",
] = (
    df.loc[
        valid_pr_reconstruction,
        "Output_Energy",
    ]
    / df.loc[
        valid_pr_reconstruction,
        "Input_Energy",
    ]
)

df["PR"] = df["PR"].replace(
    [np.inf, -np.inf],
    np.nan,
)


# ==============================================================
# 7. CYCLICAL TIME FEATURES FOR SENSOR RECONSTRUCTION
#
# These are used only for WS2 missing-value reconstruction.
# They are not autoencoder inputs.
# ==============================================================

day_of_year = df["Date"].dt.dayofyear

df["Day_Sin"] = np.sin(2 * np.pi * day_of_year / 366.0)

df["Day_Cos"] = np.cos(2 * np.pi * day_of_year / 366.0)


# ==============================================================
# 8. MULTIVARIABLE WS2 RECONSTRUCTION FUNCTION
#
# Reconstruction models are fitted only on observed training
# records. Out-of-fold validation is reported before missing
# values are reconstructed.
# ==============================================================


def reconstruct_ws2_sensor(
    dataframe,
    target,
    predictors,
):

    training_observed = (
        (dataframe["Evaluation_Set"] == "Training")
        & dataframe[target].notna()
        & dataframe[predictors].notna().all(axis=1)
    )

    missing_target = dataframe[target].isna() & dataframe[predictors].notna().all(
        axis=1
    )

    X_observed = dataframe.loc[
        training_observed,
        predictors,
    ]

    y_observed = dataframe.loc[
        training_observed,
        target,
    ]

    if len(y_observed) < 30:
        raise ValueError(f"Insufficient observed training data for {target}.")

    reconstruction_model = RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=SEED,
        n_jobs=-1,
    )

    cross_validation = KFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED,
    )

    out_of_fold_prediction = cross_val_predict(
        reconstruction_model,
        X_observed,
        y_observed,
        cv=cross_validation,
        n_jobs=-1,
    )

    reconstruction_mae = mean_absolute_error(
        y_observed,
        out_of_fold_prediction,
    )

    reconstruction_rmse = np.sqrt(
        mean_squared_error(
            y_observed,
            out_of_fold_prediction,
        )
    )

    reconstruction_r2 = r2_score(
        y_observed,
        out_of_fold_prediction,
    )

    nonzero_observed = np.abs(y_observed.to_numpy()) > 1e-9

    if nonzero_observed.any():

        reconstruction_mape = (
            np.mean(
                np.abs(
                    (
                        y_observed.to_numpy()[nonzero_observed]
                        - out_of_fold_prediction[nonzero_observed]
                    )
                    / y_observed.to_numpy()[nonzero_observed]
                )
            )
            * 100
        )

    else:
        reconstruction_mape = np.nan

    reconstruction_model.fit(
        X_observed,
        y_observed,
    )

    dataframe[f"{target}_WasMissing"] = dataframe[target].isna()

    dataframe.loc[
        missing_target,
        target,
    ] = reconstruction_model.predict(
        dataframe.loc[
            missing_target,
            predictors,
        ]
    )

    reconstruction_report = {
        "Sensor": target,
        "Observed_Training_Records": int(training_observed.sum()),
        "Reconstructed_Records": int(missing_target.sum()),
        "Validation_MAE": reconstruction_mae,
        "Validation_RMSE": reconstruction_rmse,
        "Validation_R2": reconstruction_r2,
        "Validation_MAPE_Percent": reconstruction_mape,
    }

    return reconstruction_report, reconstruction_model


# ==============================================================
# 9. RECONSTRUCT WS2 EAST AND WEST
# ==============================================================

ws2_predictors = [
    "WS1_East",
    "WS1_West",
    "Radiation_kWh",
    "Input_Energy",
    "Output_Energy",
    "Day_Sin",
    "Day_Cos",
]

east_report, east_reconstruction_model = reconstruct_ws2_sensor(
    dataframe=df,
    target="WS2_East",
    predictors=ws2_predictors,
)

west_report, west_reconstruction_model = reconstruct_ws2_sensor(
    dataframe=df,
    target="WS2_West",
    predictors=ws2_predictors,
)

reconstruction_validation = pd.DataFrame(
    [
        east_report,
        west_report,
    ]
)

reconstruction_validation.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_I_WS2_Reconstruction_Validation.xlsx",
    ),
    index=False,
)

print("\nWS2 Reconstruction Validation")
display(reconstruction_validation)


# ==============================================================
# 10. MODEL FEATURES
#
# These match the input vector stated in the research document.
# ==============================================================

model_features = [
    "WS1_East",
    "WS2_East",
    "WS1_West",
    "WS2_West",
    "Radiation_kWh",
]


# ==============================================================
# 11. ASSESSMENT ELIGIBILITY
#
# Reconstructed WS2 observations are retained in the exported
# dataset but are excluded from autoencoder training, threshold
# calibration, testing, and SPHI calculation.
#
# This prevents synthetic relationships from producing
# artificially low reconstruction errors.
# ==============================================================

df["Uses_Reconstructed_WS2"] = df["WS2_East_WasMissing"] | df["WS2_West_WasMissing"]

df["Assessment_Eligible"] = ~df["Uses_Reconstructed_WS2"] & df[
    model_features
].notna().all(axis=1)

df["Assessment_Confidence"] = np.where(
    df["Assessment_Eligible"],
    "Observed sensor inputs",
    "Not assessed: reconstructed or unavailable WS2 input",
)


# ==============================================================
# 12. DATA QUALITY REPORT
# ==============================================================

quality = pd.DataFrame(
    {
        "Parameter": [
            "Total samples",
            "Variables",
            "Duplicate dates removed",
            "Remaining missing values",
            "Assessment-eligible samples",
            "Excluded reconstructed-input samples",
            "Start date",
            "End date",
            "PR reconstructed observations",
        ],
        "Value": [
            len(df),
            len(df.columns),
            number_of_rows - len(df),
            df.isna().sum().sum(),
            df["Assessment_Eligible"].sum(),
            df["Uses_Reconstructed_WS2"].sum(),
            df["Date"].min(),
            df["Date"].max(),
            df["PR_WasMissing"].sum(),
        ],
    }
)

quality.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_II_Data_Quality.xlsx",
    ),
    index=False,
)

display(quality)


# ==============================================================
# 13. FEATURE STATISTICS
# ==============================================================

df.loc[
    df["Assessment_Eligible"],
    model_features,
].describe().T.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_III_Feature_Statistics.xlsx",
    )
)


# ==============================================================
# 14. CREATE RAW CHRONOLOGICAL DATASETS
# ==============================================================

training_mask = (df["Evaluation_Set"] == "Training") & df["Assessment_Eligible"]

calibration_mask = (df["Evaluation_Set"] == "Threshold Calibration") & df[
    "Assessment_Eligible"
]

test_mask = (df["Evaluation_Set"] == "Unseen Test") & df["Assessment_Eligible"]

X_training_candidate = df.loc[
    training_mask,
    model_features,
].copy()

X_calibration = df.loc[
    calibration_mask,
    model_features,
].copy()

X_test = df.loc[
    test_mask,
    model_features,
].copy()

if len(X_training_candidate) == 0:
    raise ValueError("No eligible training observations are available.")

if len(X_calibration) == 0:
    raise ValueError("No eligible calibration observations are available.")

if len(X_test) == 0:
    raise ValueError("No eligible unseen test observations are available.")


# ==============================================================
# 15. UNSUPERVISED NORMAL-REFERENCE SCREENING
#
# The paper states that the autoencoder learns normal operating
# behaviour. Because fault labels are unavailable, Isolation
# Forest is applied only within the training partition to remove
# the most unusual 10% of training observations.
# ==============================================================

normal_screening_scaler = MinMaxScaler()

X_training_screened_scaled = normal_screening_scaler.fit_transform(X_training_candidate)

normal_screening_model = IsolationForest(
    n_estimators=500,
    contamination=NORMAL_CONTAMINATION,
    random_state=SEED,
    n_jobs=-1,
)

normal_screening_labels = normal_screening_model.fit_predict(X_training_screened_scaled)

normal_training_mask = normal_screening_labels == 1

X_train_normal = X_training_candidate.loc[normal_training_mask].copy()

if len(X_train_normal) < 30:
    raise ValueError("Too few normal-reference training observations.")

df["Normal_Reference_Training"] = False

df.loc[
    X_train_normal.index,
    "Normal_Reference_Training",
] = True


# ==============================================================
# 16. DATASET SPLIT REPORT
# ==============================================================

split_table = pd.DataFrame(
    {
        "Dataset": [
            "Training candidates",
            "Normal-reference training",
            "Threshold calibration",
            "Unseen test",
            "Excluded reconstructed-input records",
        ],
        "Samples": [
            len(X_training_candidate),
            len(X_train_normal),
            len(X_calibration),
            len(X_test),
            int(df["Uses_Reconstructed_WS2"].sum()),
        ],
        "Purpose": [
            "Unsupervised normal-reference screening",
            "Scaler and autoencoder fitting",
            "MAD threshold and SPHI calibration",
            "Independent model assessment",
            "Retained for audit but not health scoring",
        ],
    }
)

split_table.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_IV_Data_Split.xlsx",
    ),
    index=False,
)

display(split_table)


# ==============================================================
# 17. MIN-MAX SCALING
#
# The scaler is fitted only on normal-reference training data.
# ==============================================================

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train_normal)

X_calibration_scaled = scaler.transform(X_calibration)

X_test_scaled = scaler.transform(X_test)

if not np.isfinite(X_train_scaled).all():
    raise ValueError("Non-finite values found in scaled training data.")

if not np.isfinite(X_calibration_scaled).all():
    raise ValueError("Non-finite values found in scaled calibration data.")

if not np.isfinite(X_test_scaled).all():
    raise ValueError("Non-finite values found in scaled test data.")


# ==============================================================
# 18. AUTOENCODER
#
# Architecture retained from the research document:
#
# 5 -> 16 -> 4 -> 16 -> 5
# ==============================================================

inputs = Input(
    shape=(len(model_features),),
    name="Operational_Input",
)

x = Dense(
    16,
    activation="relu",
    name="Encoder",
)(inputs)

x = Dropout(
    0.10,
    name="Training_Dropout",
)(x)

latent = Dense(
    4,
    activation="relu",
    name="Latent_Space",
)(x)

x = Dense(
    16,
    activation="relu",
    name="Decoder",
)(latent)

outputs = Dense(
    len(model_features),
    activation="linear",
    name="Reconstructed_Input",
)(x)

autoencoder = Model(
    inputs=inputs,
    outputs=outputs,
    name="SPHI_Autoencoder",
)

autoencoder.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss="mse",
)

autoencoder.summary()


# ==============================================================
# 19. MODEL CONFIGURATION
# ==============================================================

model_configuration = pd.DataFrame(
    {
        "Parameter": [
            "Input features",
            "Architecture",
            "Latent dimension",
            "Scaler",
            "Optimizer",
            "Learning rate",
            "Loss",
            "Dropout",
            "Normal-reference screening",
            "Training fraction",
            "Calibration fraction",
            "Test fraction",
        ],
        "Value": [
            str(model_features),
            "5-16-4-16-5",
            4,
            "MinMaxScaler",
            "Adam",
            0.0005,
            "MSE",
            0.10,
            ("Isolation Forest; " f"contamination={NORMAL_CONTAMINATION}"),
            TRAIN_FRACTION,
            CALIBRATION_FRACTION,
            TEST_FRACTION,
        ],
    }
)

model_configuration.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_V_Model_Configuration.xlsx",
    ),
    index=False,
)


# ==============================================================
# 20. TRAINING CALLBACKS
#
# Internal validation is created only from normal-reference
# training records. The threshold calibration partition remains
# untouched during model fitting.
# ==============================================================

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=50,
    restore_best_weights=True,
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=15,
    min_lr=1e-7,
)


# ==============================================================
# 21. AUTOENCODER TRAINING
# ==============================================================

history = autoencoder.fit(
    X_train_scaled,
    X_train_scaled,
    validation_split=0.15,
    epochs=500,
    batch_size=16,
    shuffle=False,
    callbacks=[
        early_stop,
        reduce_lr,
    ],
    verbose=1,
)


# ==============================================================
# 22. TRAINING PERFORMANCE
# ==============================================================

training_table = pd.DataFrame(
    {
        "Parameter": [
            "Completed epochs",
            "Minimum training loss",
            "Minimum internal validation loss",
        ],
        "Value": [
            len(history.history["loss"]),
            min(history.history["loss"]),
            min(history.history["val_loss"]),
        ],
    }
)

training_table.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_VI_Training_Performance.xlsx",
    ),
    index=False,
)

display(training_table)


# ==============================================================
# 23. TRAINING CURVE
# ==============================================================

plt.figure(
    figsize=(8, 5),
    dpi=300,
)

plt.plot(
    history.history["loss"],
    linewidth=2,
    label="Training loss",
)

plt.plot(
    history.history["val_loss"],
    linewidth=2,
    label="Internal validation loss",
)

plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Autoencoder Training Performance")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        FIG_DIR,
        "Figure_01_Training_Curve.png",
    ),
    dpi=300,
    bbox_inches="tight",
)

plt.close()


# ==============================================================
# 24. RECONSTRUCTION FUNCTION
# ==============================================================


def calculate_reconstruction(
    model,
    scaled_data,
):

    prediction = model.predict(
        scaled_data,
        verbose=0,
    )

    error = np.mean(
        np.square(scaled_data - prediction),
        axis=1,
    )

    return prediction, error


# ==============================================================
# 25. CALIBRATION RECONSTRUCTION
#
# Calibration data were not used for scaler fitting,
# autoencoder fitting, or early stopping.
# ==============================================================

(
    X_calibration_prediction,
    calibration_error,
) = calculate_reconstruction(
    autoencoder,
    X_calibration_scaled,
)


# ==============================================================
# 26. MAD THRESHOLD CALIBRATION
# ==============================================================

calibration_median = np.median(calibration_error)

calibration_mad = np.median(np.abs(calibration_error - calibration_median))

threshold = calibration_median + MAD_MULTIPLIER * calibration_mad

if not np.isfinite(threshold) or threshold <= 0:
    raise ValueError("The calibrated MAD threshold is invalid.")

threshold_table = pd.DataFrame(
    {
        "Parameter": [
            "Calibration median error",
            "Calibration MAD",
            "MAD multiplier",
            "Final threshold",
        ],
        "Value": [
            calibration_median,
            calibration_mad,
            MAD_MULTIPLIER,
            threshold,
        ],
    }
)

threshold_table.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_VII_Robust_Threshold.xlsx",
    ),
    index=False,
)

display(threshold_table)


# ==============================================================
# 27. CALIBRATION ERROR DISTRIBUTION
# ==============================================================

plt.figure(
    figsize=(7, 5),
    dpi=300,
)

plt.hist(
    calibration_error,
    bins=20,
)

plt.axvline(
    threshold,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Median + 3MAD",
)

plt.xlabel("Reconstruction Error")
plt.ylabel("Frequency")
plt.title("Threshold Calibration Error Distribution")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        FIG_DIR,
        "Figure_02_Calibration_Error_Distribution.png",
    ),
    dpi=300,
    bbox_inches="tight",
)

plt.close()


# ==============================================================
# 28. INDEPENDENT TEST RECONSTRUCTION
# ==============================================================

(
    X_test_prediction,
    test_error,
) = calculate_reconstruction(
    autoencoder,
    X_test_scaled,
)


# ==============================================================
# 29. INDEPENDENT TEST RECONSTRUCTION METRICS
# ==============================================================

feature_results = []

for feature_index, feature in enumerate(model_features):

    feature_mse = mean_squared_error(
        X_test_scaled[:, feature_index],
        X_test_prediction[:, feature_index],
    )

    feature_rmse = np.sqrt(feature_mse)

    feature_mae = mean_absolute_error(
        X_test_scaled[:, feature_index],
        X_test_prediction[:, feature_index],
    )

    feature_r2 = r2_score(
        X_test_scaled[:, feature_index],
        X_test_prediction[:, feature_index],
    )

    feature_results.append(
        {
            "Feature": feature,
            "MSE": feature_mse,
            "RMSE": feature_rmse,
            "MAE": feature_mae,
            "R2": feature_r2,
        }
    )

feature_table = pd.DataFrame(feature_results)

overall_mse = mean_squared_error(
    X_test_scaled,
    X_test_prediction,
)

overall_rmse = np.sqrt(overall_mse)

overall_mae = mean_absolute_error(
    X_test_scaled,
    X_test_prediction,
)

overall_r2 = r2_score(
    X_test_scaled,
    X_test_prediction,
    multioutput="uniform_average",
)

performance_table = pd.DataFrame(
    {
        "Metric": [
            "Independent test MSE",
            "Independent test RMSE",
            "Independent test MAE",
            "Independent test mean R2",
        ],
        "Value": [
            overall_mse,
            overall_rmse,
            overall_mae,
            overall_r2,
        ],
    }
)

performance_table.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_VIII_Test_Reconstruction_Performance.xlsx",
    ),
    index=False,
)

feature_table.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_IX_Featurewise_Test_Performance.xlsx",
    ),
    index=False,
)

print("\nIndependent Test Performance")
display(performance_table)

print("\nFeature-wise Test Performance")
display(feature_table)


# ==============================================================
# 30. RECONSTRUCT ALL ELIGIBLE OBSERVATIONS
#
# Rows containing reconstructed WS2 values are not passed to
# the autoencoder.
# ==============================================================

eligible_indices = df.index[df["Assessment_Eligible"]]

X_all_eligible = df.loc[
    eligible_indices,
    model_features,
]

X_all_eligible_scaled = scaler.transform(X_all_eligible)

(
    X_all_prediction,
    full_error,
) = calculate_reconstruction(
    autoencoder,
    X_all_eligible_scaled,
)

results = df.copy()

results["Reconstruction_Error"] = np.nan
results["Threshold"] = threshold

results.loc[
    eligible_indices,
    "Reconstruction_Error",
] = full_error


# ==============================================================
# 31. CANDIDATE OPERATIONAL DEVIATION IDENTIFICATION
#
# The model detects deviations, not confirmed equipment faults.
# ==============================================================

results["Deviation_Status"] = "Not Assessed"

eligible_deviation = (
    results.loc[
        eligible_indices,
        "Reconstruction_Error",
    ]
    > threshold
)

results.loc[
    eligible_indices,
    "Deviation_Status",
] = np.where(
    eligible_deviation,
    "Candidate Operational Deviation",
    "Normal",
)


# ==============================================================
# 32. SPHI CALIBRATION
#
# This equation matches the Word document:
#
# SPHI = 100 * (
#     1 - (E_i - E_min) / (E_max - E_min)
# )
#
# E_min and E_max are learned only from the independent
# calibration distribution, preventing test-data leakage.
# ==============================================================

calibration_error_min = np.min(calibration_error)

calibration_error_max = np.max(calibration_error)

calibration_error_range = calibration_error_max - calibration_error_min

if calibration_error_range <= 0:
    raise ValueError("Calibration errors have no usable range.")

eligible_errors = results.loc[
    eligible_indices,
    "Reconstruction_Error",
]

eligible_sphi = 100 * (
    1 - (eligible_errors - calibration_error_min) / calibration_error_range
)

eligible_sphi = np.clip(
    eligible_sphi,
    0,
    100,
)

results["SPHI"] = np.nan

results.loc[
    eligible_indices,
    "SPHI",
] = eligible_sphi


# ==============================================================
# 33. HEALTH CLASSIFICATION
# ==============================================================


def health_class(
    value,
):

    if pd.isna(value):
        return "Not Assessed"

    if value >= 80:
        return "Healthy"

    if value >= 60:
        return "Warning"

    return "Degraded"


results["Health_Status"] = results["SPHI"].apply(health_class)


# ==============================================================
# 34. SPHI METHODOLOGY TABLE
# ==============================================================

sphi_method = pd.DataFrame(
    {
        "Parameter": [
            "SPHI equation",
            "SPHI calibration population",
            "Threshold method",
            "Healthy range",
            "Warning range",
            "Degraded range",
            "Unassessed records",
        ],
        "Value": [
            ("100*(1-(Error-Emin)/(Emax-Emin)); " "clipped to [0,100]"),
            "Independent threshold-calibration partition",
            "Calibration median + 3MAD",
            "SPHI >= 80",
            "60 <= SPHI < 80",
            "SPHI < 60",
            ("Records containing reconstructed or " "unavailable WS2 inputs"),
        ],
    }
)

sphi_method.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_X_SPHI_Methodology.xlsx",
    ),
    index=False,
)


# ==============================================================
# 35. DEVIATION SUMMARY
# ==============================================================

assessed_results = results[results["Assessment_Eligible"]].copy()

candidate_deviation_count = (
    assessed_results["Deviation_Status"] == "Candidate Operational Deviation"
).sum()

normal_count = (assessed_results["Deviation_Status"] == "Normal").sum()

deviation_summary = pd.DataFrame(
    {
        "Parameter": [
            "Total source observations",
            "Assessed observations",
            "Unassessed reconstructed-input observations",
            "Normal observations",
            "Candidate operational deviations",
            "Candidate deviation percentage",
        ],
        "Value": [
            len(results),
            len(assessed_results),
            (results["Health_Status"] == "Not Assessed").sum(),
            normal_count,
            candidate_deviation_count,
            (candidate_deviation_count / len(assessed_results) * 100),
        ],
    }
)

deviation_summary.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_XI_Deviation_Summary.xlsx",
    ),
    index=False,
)

display(deviation_summary)


# ==============================================================
# 36. SPHI SUMMARY
# ==============================================================

sphi_summary = pd.DataFrame(
    {
        "Parameter": [
            "Average SPHI",
            "Maximum SPHI",
            "Minimum SPHI",
            "Healthy observations",
            "Warning observations",
            "Degraded observations",
            "Not-assessed observations",
        ],
        "Value": [
            assessed_results["SPHI"].mean(),
            assessed_results["SPHI"].max(),
            assessed_results["SPHI"].min(),
            (results["Health_Status"] == "Healthy").sum(),
            (results["Health_Status"] == "Warning").sum(),
            (results["Health_Status"] == "Degraded").sum(),
            (results["Health_Status"] == "Not Assessed").sum(),
        ],
    }
)

sphi_summary.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_XII_SPHI_Summary.xlsx",
    ),
    index=False,
)

display(sphi_summary)


# ==============================================================
# 37. RECONSTRUCTION ERROR TIME SERIES
# ==============================================================

plt.figure(
    figsize=(12, 5),
    dpi=300,
)

plt.plot(
    assessed_results["Date"],
    assessed_results["Reconstruction_Error"],
    linewidth=1.5,
    label="Reconstruction error",
)

plt.axhline(
    threshold,
    linestyle="--",
    linewidth=2,
    color="red",
    label="Calibration median + 3MAD",
)

candidate_points = assessed_results[
    assessed_results["Deviation_Status"] == "Candidate Operational Deviation"
]

plt.scatter(
    candidate_points["Date"],
    candidate_points["Reconstruction_Error"],
    s=35,
    color="red",
    label="Candidate operational deviation",
)

plt.xlabel("Date")
plt.ylabel("Reconstruction Error")
plt.title("PV Operational Deviation Detection")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        FIG_DIR,
        "Figure_03_Reconstruction_Error.png",
    ),
    dpi=300,
    bbox_inches="tight",
)

plt.close()


# ==============================================================
# 38. SPHI TIME SERIES
# ==============================================================

plt.figure(
    figsize=(12, 5),
    dpi=300,
)

plt.plot(
    assessed_results["Date"],
    assessed_results["SPHI"],
    linewidth=2,
    label="SPHI",
)

plt.axhline(
    80,
    linestyle="--",
    linewidth=1.5,
    color="orange",
    label="Healthy/Warning boundary",
)

plt.axhline(
    60,
    linestyle="--",
    linewidth=1.5,
    color="red",
    label="Warning/Degraded boundary",
)

plt.xlabel("Date")
plt.ylabel("SPHI")
plt.title("Solar Plant Health Index Variation")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        FIG_DIR,
        "Figure_04_SPHI_Time_Series.png",
    ),
    dpi=300,
    bbox_inches="tight",
)

plt.close()


# ==============================================================
# 39. MONTHLY ANALYSIS
# ==============================================================

if "Month" not in results.columns:
    results["Month"] = results["Date"].dt.month_name()

if "Month_Number" not in results.columns:
    results["Month_Number"] = results["Date"].dt.month

monthly_analysis = (
    results[results["Assessment_Eligible"]]
    .groupby(
        [
            "Month_Number",
            "Month",
        ],
        as_index=False,
    )
    .agg(
        Mean_SPHI=(
            "SPHI",
            "mean",
        ),
        Median_SPHI=(
            "SPHI",
            "median",
        ),
        Mean_PR=(
            "PR",
            "mean",
        ),
        Mean_Error=(
            "Reconstruction_Error",
            "mean",
        ),
        Healthy_Observations=(
            "Health_Status",
            lambda values: (values == "Healthy").sum(),
        ),
        Warning_Observations=(
            "Health_Status",
            lambda values: (values == "Warning").sum(),
        ),
        Degraded_Observations=(
            "Health_Status",
            lambda values: (values == "Degraded").sum(),
        ),
    )
    .sort_values("Month_Number")
)

monthly_analysis.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_XIII_Monthly_SPHI_Analysis.xlsx",
    ),
    index=False,
)

display(monthly_analysis)


# ==============================================================
# 40. MONTHLY SPHI FIGURE
# ==============================================================

plt.figure(
    figsize=(10, 5),
    dpi=300,
)

plt.bar(
    monthly_analysis["Month"],
    monthly_analysis["Mean_SPHI"],
)

plt.axhline(
    80,
    linestyle="--",
    linewidth=1.5,
    color="orange",
    label="Healthy/Warning boundary",
)

plt.axhline(
    60,
    linestyle="--",
    linewidth=1.5,
    color="red",
    label="Warning/Degraded boundary",
)

plt.xlabel("Month")
plt.ylabel("Average SPHI")
plt.title("Monthly Solar Plant Health Index")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        FIG_DIR,
        "Figure_05_Monthly_SPHI.png",
    ),
    dpi=300,
    bbox_inches="tight",
)

plt.close()


# ==============================================================
# 41. ENERGY YIELD
# ==============================================================

results["Energy_Yield"] = np.where(
    results["Radiation_kWh"] > 0,
    (results["Output_Energy"] / results["Radiation_kWh"]),
    np.nan,
)

results["Energy_Yield"] = results["Energy_Yield"].replace(
    [np.inf, -np.inf],
    np.nan,
)


# ==============================================================
# 42. INDEPENDENT-INDICATOR VALIDATION
# ==============================================================

validation_pr = results[
    results["Assessment_Eligible"] & results["SPHI"].notna() & results["PR"].notna()
].copy()

validation_energy = results[
    results["Assessment_Eligible"]
    & results["SPHI"].notna()
    & results["Energy_Yield"].notna()
].copy()

if len(validation_pr) >= 3:

    corr_pr, p_pr = pearsonr(
        validation_pr["SPHI"],
        validation_pr["PR"],
    )

else:
    corr_pr = np.nan
    p_pr = np.nan

if len(validation_energy) >= 3:

    corr_energy, p_energy = pearsonr(
        validation_energy["SPHI"],
        validation_energy["Energy_Yield"],
    )

else:
    corr_energy = np.nan
    p_energy = np.nan

validation_table = pd.DataFrame(
    {
        "Relationship": [
            "SPHI vs PR",
            "SPHI vs Energy Yield",
        ],
        "Pearson_R": [
            corr_pr,
            corr_energy,
        ],
        "p_value": [
            p_pr,
            p_energy,
        ],
        "Valid_Observations": [
            len(validation_pr),
            len(validation_energy),
        ],
    }
)

validation_table.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_XIV_SPHI_Validation.xlsx",
    ),
    index=False,
)

display(validation_table)


# ==============================================================
# 43. VALIDATION FIGURES
# ==============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4),
    dpi=300,
)

axes[0].scatter(
    validation_pr["SPHI"],
    validation_pr["PR"],
    s=25,
)

axes[0].set_xlabel("SPHI")
axes[0].set_ylabel("Performance Ratio")
axes[0].set_title(f"SPHI vs PR (r={corr_pr:.3f})")
axes[0].grid(alpha=0.3)

axes[1].scatter(
    validation_energy["SPHI"],
    validation_energy["Energy_Yield"],
    s=25,
)

axes[1].set_xlabel("SPHI")
axes[1].set_ylabel("Energy Yield")
axes[1].set_title(f"SPHI vs Energy Yield (r={corr_energy:.3f})")
axes[1].grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIG_DIR,
        "Figure_06_SPHI_Validation.png",
    ),
    dpi=300,
    bbox_inches="tight",
)

plt.close()


# ==============================================================
# 44. CALIBRATION-BASED THRESHOLD SENSITIVITY
#
# Every alternative threshold is calculated using the same
# independent calibration-error distribution.
# ==============================================================

threshold_results = []

for multiplier in [
    2,
    3,
    4,
]:

    sensitivity_threshold = calibration_median + multiplier * calibration_mad

    sensitivity_deviation_count = (
        assessed_results["Reconstruction_Error"] > sensitivity_threshold
    ).sum()

    threshold_results.append(
        {
            "MAD_Multiplier": multiplier,
            "Calibration_Median": calibration_median,
            "Calibration_MAD": calibration_mad,
            "Threshold": sensitivity_threshold,
            "Candidate_Deviation_Observations": (sensitivity_deviation_count),
            "Candidate_Deviation_Percentage": (
                sensitivity_deviation_count / len(assessed_results) * 100
            ),
        }
    )

threshold_sensitivity = pd.DataFrame(threshold_results)

threshold_sensitivity.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_XV_Threshold_Sensitivity.xlsx",
    ),
    index=False,
)

display(threshold_sensitivity)


# ==============================================================
# 45. CANDIDATE OPERATIONAL DEVIATION EVENTS
#
# These are not called confirmed degradation events because no
# maintenance/fault labels are available.
# ==============================================================

candidate_events = (
    assessed_results[
        assessed_results["Deviation_Status"] == "Candidate Operational Deviation"
    ]
    .sort_values(
        "Reconstruction_Error",
        ascending=False,
    )
    .head(20)
)

candidate_events[
    [
        "Date",
        "Evaluation_Set",
        "SPHI",
        "Reconstruction_Error",
        "PR",
        "Radiation_kWh",
        "Output_Energy",
        "Health_Status",
        "Deviation_Status",
    ]
].to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_XVI_Candidate_Operational_Deviations.xlsx",
    ),
    index=False,
)


# ==============================================================
# 46. UNSEEN TEST SUMMARY
# ==============================================================

test_results = results[test_mask].copy()

test_candidate_count = (
    test_results["Deviation_Status"] == "Candidate Operational Deviation"
).sum()

test_summary = pd.DataFrame(
    {
        "Metric": [
            "Test observations",
            "Test candidate deviations",
            "Test candidate deviation percentage",
            "Mean test SPHI",
            "Median test SPHI",
            "Minimum test SPHI",
            "Test reconstruction RMSE",
            "Test reconstruction MAE",
            "Test reconstruction R2",
        ],
        "Value": [
            len(test_results),
            test_candidate_count,
            (test_candidate_count / len(test_results) * 100),
            test_results["SPHI"].mean(),
            test_results["SPHI"].median(),
            test_results["SPHI"].min(),
            overall_rmse,
            overall_mae,
            overall_r2,
        ],
    }
)

test_summary.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_XVII_Unseen_Test_Performance.xlsx",
    ),
    index=False,
)

display(test_summary)


# ==============================================================
# 47. IEEE-PAPER SUMMARY TABLES
# ==============================================================

paper_model_table = pd.DataFrame(
    {
        "Parameter": [
            "Architecture",
            "Input features",
            "Latent dimension",
            "Optimizer",
            "Loss function",
            "Independent test RMSE",
            "Independent test MAE",
            "Independent test R2",
        ],
        "Value": [
            "5-16-4-16-5",
            len(model_features),
            4,
            "Adam",
            "MSE",
            round(overall_rmse, 5),
            round(overall_mae, 5),
            round(overall_r2, 5),
        ],
    }
)

paper_health_table = pd.DataFrame(
    {
        "Parameter": [
            "Average SPHI",
            "Maximum SPHI",
            "Minimum SPHI",
            "Healthy observations",
            "Warning observations",
            "Degraded observations",
            "Not-assessed observations",
        ],
        "Value": [
            round(
                assessed_results["SPHI"].mean(),
                3,
            ),
            round(
                assessed_results["SPHI"].max(),
                3,
            ),
            round(
                assessed_results["SPHI"].min(),
                3,
            ),
            (results["Health_Status"] == "Healthy").sum(),
            (results["Health_Status"] == "Warning").sum(),
            (results["Health_Status"] == "Degraded").sum(),
            (results["Health_Status"] == "Not Assessed").sum(),
        ],
    }
)

print("\nTABLE I")
print("Autoencoder Configuration and " "Independent Test Performance")
display(paper_model_table)

print("\nTABLE II")
print("Threshold Sensitivity Analysis")
display(threshold_sensitivity)

print("\nTABLE III")
print("SPHI Health Assessment")
display(paper_health_table)

print("\nTABLE IV")
print("Monthly SPHI Performance")
display(monthly_analysis)

print("\nTABLE V")
print("Candidate Operational Deviation Events")
display(
    candidate_events[
        [
            "Date",
            "SPHI",
            "Reconstruction_Error",
            "PR",
            "Output_Energy",
            "Health_Status",
        ]
    ].head(10)
)


# ==============================================================
# 48. SAVE MODELS AND PREPROCESSORS
# ==============================================================

autoencoder.save(
    os.path.join(
        RESULT_DIR,
        "SPHI_Autoencoder_Model.keras",
    )
)

joblib.dump(
    scaler,
    os.path.join(
        RESULT_DIR,
        "SPHI_MinMaxScaler.joblib",
    ),
)

joblib.dump(
    normal_screening_model,
    os.path.join(
        RESULT_DIR,
        "Normal_Reference_IsolationForest.joblib",
    ),
)

joblib.dump(
    east_reconstruction_model,
    os.path.join(
        RESULT_DIR,
        "WS2_East_Reconstruction_Model.joblib",
    ),
)

joblib.dump(
    west_reconstruction_model,
    os.path.join(
        RESULT_DIR,
        "WS2_West_Reconstruction_Model.joblib",
    ),
)


# ==============================================================
# 49. FINAL EXPORT
# ==============================================================

results.drop(
    columns=[
        "Day_Sin",
        "Day_Cos",
    ],
    errors="ignore",
).to_excel(
    os.path.join(
        RESULT_DIR,
        "FINAL_SPHI_RESULTS.xlsx",
    ),
    index=False,
)

parameters = pd.DataFrame(
    {
        "Parameter": [
            "Random seed",
            "Architecture",
            "Input features",
            "Training fraction",
            "Calibration fraction",
            "Test fraction",
            "Normal-reference contamination",
            "Threshold method",
            "MAD multiplier",
            "SPHI equation",
            "SPHI calibration population",
            "Healthy threshold",
            "Warning threshold",
        ],
        "Value": [
            SEED,
            "5-16-4-16-5",
            str(model_features),
            TRAIN_FRACTION,
            CALIBRATION_FRACTION,
            TEST_FRACTION,
            NORMAL_CONTAMINATION,
            "Calibration Median + MAD",
            MAD_MULTIPLIER,
            ("100*(1-(Error-Emin)/(Emax-Emin)); " "clipped to [0,100]"),
            "Independent calibration partition",
            80,
            60,
        ],
    }
)

parameters.to_excel(
    os.path.join(
        TABLE_DIR,
        "Table_XVIII_Final_Model_Parameters.xlsx",
    ),
    index=False,
)


# ==============================================================
# 50. COMPLETION MESSAGE
# ==============================================================

print("\n==================================================")

print("CORRECTED SPHI ANALYSIS COMPLETED SUCCESSFULLY")

print("==================================================")

print(f"Total source observations: {len(results)}")

print(
    "Assessment-eligible observations:",
    int(results["Assessment_Eligible"].sum()),
)

print(
    "Not-assessed reconstructed-input observations:",
    int((results["Health_Status"] == "Not Assessed").sum()),
)

print(
    "Independent test R2:",
    round(overall_r2, 5),
)

print(
    "Final MAD threshold:",
    threshold,
)

print(
    "Results directory:",
    RESULT_DIR,
)